# Étape 1 — Installation de l'environnement
Clonage de `finetune-hf-vits`, installation des dépendances, mise à jour de `huggingface_hub`.

In [ ]:
!git clone https://github.com/ylacombe/finetune-hf-vits.git
%cd finetune-hf-vits
!pip install -r requirements.txt
!pip install huggingface_hub -U

# Étape 2 — Compilation de l'alignement monotone et connexion à Hugging Face
Compile l'extension Cython nécessaire à l'entraînement VITS, puis authentifie la session (`notebook_login`) pour pouvoir pousser des dépôts sur le Hub.

In [ ]:
!cd monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Étape 3 — Exploration : dataset et répartition des locuteurs (brouillon)
**Étape exploratoire.** Charge `bau_tts`, compte les échantillons par locuteur (`speaker_id`) sur le train, et effectue un premier filtrage brut (texte **non nettoyé**) — remplacée par la version finale nettoyée de l'Étape 6.

In [ ]:
from datasets import load_dataset, DatasetDict
from collections import Counter

train_ds = load_dataset("google/WaxalNLP", "bau_tts", split="train")
val_ds   = load_dataset("google/WaxalNLP", "bau_tts", split="validation")
test_ds  = load_dataset("google/WaxalNLP", "bau_tts", split="test")

print(train_ds)
print(train_ds[0])

counts = Counter(train_ds["speaker_id"])
print(counts.most_common())

In [ ]:
best_speaker = counts.most_common(1)[0][0]
print("Locuteur choisi :", best_speaker, "-", counts[best_speaker], "échantillons train")

def keep_speaker(example):
    return example["speaker_id"] == best_speaker

mono_train = train_ds.filter(keep_speaker)
mono_val   = val_ds.filter(keep_speaker)
mono_test  = test_ds.filter(keep_speaker)

print(len(mono_train), len(mono_val), len(mono_test))

In [ ]:
import numpy as np

durations = []
for ex in mono_train:
    arr = ex["audio"].get_all_samples()  # ou ex["audio"]["array"] selon la version de `datasets`
    sr = arr.sample_rate if hasattr(arr, "sample_rate") else ex["audio"]["sampling_rate"]
    n_samples = arr.data.shape[-1] if hasattr(arr, "data") else len(ex["audio"]["array"])
    durations.append(n_samples / sr)

durations = np.array(durations)
print("min :", durations.min(), "max :", durations.max())
print("médiane :", np.median(durations))
print("nb échantillons ≤ 20s :", (durations <= 20).sum(), "/", len(durations))
print("nb échantillons ≤ 30s :", (durations <= 30).sum(), "/", len(durations))

# Étape 4 — Exploration : vocabulaire baoulé brut vs vocabulaire akan
**Étape exploratoire** (texte non nettoyé). Compare les caractères présents dans `bau_tts` à ceux du tokenizer `facebook/mms-tts-aka`, pour visualiser l'écart avant nettoyage.

In [ ]:
from datasets import load_dataset

full_bau = load_dataset("google/WaxalNLP", "bau_tts", split="train+validation+test")
all_text = " ".join(full_bau["text"]).lower()

baoule_chars = sorted(set(all_text))
print("Caractères baoulé trouvés :", baoule_chars)
print("Nombre total :", len(baoule_chars))

aka_chars = set("a ʼ t - n ' _ 3 p m á w y ɛ f o g u k h l s 2 e r i ɔ d b".split())
baoule_set = set(baoule_chars)

manquants = baoule_set - aka_chars
print("\nCaractères présents en baoulé mais ABSENTS du vocab akan :")
print(sorted(manquants))

# Étape 6 — Version finale : nettoyage du texte, filtrage sur le locuteur JH, publication du dataset
Fonction `clean_baoule_text` (normalise apostrophes/guillemets/tirets, retire emoji et bruit typographique — garde les vrais phonèmes, noms propres et nombres entre crochets prononcés). Filtre le dataset sur le locuteur le plus représenté (JH, 276 échantillons train) et publie le résultat sur `bau-tts-monospeaker`.

In [ ]:
import unicodedata, re
from datasets import load_dataset, DatasetDict
from collections import Counter

def clean_baoule_text(text):
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u200b", "").replace("\xa0", " ")
    text = re.sub(r"[’‘]", "'", text)
    text = re.sub(r"[«»“”]", '"', text)
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", "", text)
    text = text.replace("°", "").replace("²", "").replace("•", "").replace("→", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_ds = load_dataset("google/WaxalNLP", "bau_tts", split="train")
val_ds   = load_dataset("google/WaxalNLP", "bau_tts", split="validation")
test_ds  = load_dataset("google/WaxalNLP", "bau_tts", split="test")

# Nettoyage du texte SANS passer par .map() (évite l'overflow PyArrow sur la colonne audio)
def clean_split(ds):
    cleaned_texts = [clean_baoule_text(t) for t in ds["text"]]
    ds = ds.remove_columns("text").add_column("text", cleaned_texts)
    return ds

train_ds = clean_split(train_ds)
val_ds   = clean_split(val_ds)
test_ds  = clean_split(test_ds)

print("Nettoyage OK :", train_ds[0]["text"][:100])

best_speaker = "JH"

def keep_speaker(example):
    return example["speaker_id"] == best_speaker

mono_train = train_ds.filter(keep_speaker, writer_batch_size=50)
mono_val   = val_ds.filter(keep_speaker, writer_batch_size=50)
mono_test  = test_ds.filter(keep_speaker, writer_batch_size=50)

print(len(mono_train), len(mono_val), len(mono_test))

mono_ds = DatasetDict({"train": mono_train, "validation": mono_val, "test": mono_test})

TON_PSEUDO = "TON-PSEUDO-HF"
mono_ds.push_to_hub(f"{TON_PSEUDO}/bau-tts-monospeaker")

# Étape 7 — Version finale : construction et publication du tokenizer baoulé complet
Reconstruit le vocabulaire à partir du texte **nettoyé** (tout `bau_tts`, pas seulement JH), avec `<pad>` (id 0, token blanc VITS) et `<unk>` en tokens spéciaux dédiés. Publie sur `mms-tts-bau-tokenizer` (écrase la version brouillon de l'Étape 5).

In [ ]:
import json, os
from huggingface_hub import HfApi

# Le Vocabulaire à été construit sur TOUT bau_tts nettoyé (pas juste JH), pour être complet
full_bau_all = load_dataset("google/WaxalNLP", "bau_tts", split="train+validation+test")
cleaned_all_text = " ".join(clean_baoule_text(t) for t in full_bau_all["text"]).lower()
baoule_chars = sorted(set(cleaned_all_text))

print("Vocabulaire baoulé final :", baoule_chars)
print("Taille :", len(baoule_chars))

vocab = {"<pad>": 0}
for i, ch in enumerate(baoule_chars, start=1):
    vocab[ch] = i
vocab["<unk>"] = len(vocab)

os.makedirs("bau_tokenizer", exist_ok=True)
with open("bau_tokenizer/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

tokenizer_config = {
    "add_blank": True,
    "clean_up_tokenization_spaces": True,
    "is_uroman": False,
    "language": "bau",
    "normalize": True,
    "phonemize": False,
    "pad_token": "<pad>",
    "unk_token": "<unk>",
    "tokenizer_class": "VitsTokenizer",
}
with open("bau_tokenizer/tokenizer_config.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer_config, f, ensure_ascii=False, indent=2)

with open("bau_tokenizer/special_tokens_map.json", "w", encoding="utf-8") as f:
    json.dump({"pad_token": "<pad>", "unk_token": "<unk>"}, f, ensure_ascii=False, indent=2)

api = HfApi()
api.create_repo(f"{TON_PSEUDO}/mms-tts-bau-tokenizer", exist_ok=True)
api.upload_folder(folder_path="bau_tokenizer", repo_id=f"{TON_PSEUDO}/mms-tts-bau-tokenizer")

# Étape 8 — Configuration finale du fine-tuning (fichier JSON)
Assemble tous les paramètres d'entraînement (dataset nettoyé, tokenizer étendu avec `override_vocabulary_embeddings: true`, hyperparamètres VITS/GAN) dans `finetune_baoule.json`, consommé par `run_vits_finetuning.py`.

In [ ]:
config = {
    "project_name": "mms_baoule_finetuning",
    "push_to_hub": True,
    "hub_model_id": "TON-PSEUDO-HF/mms-tts-bau-finetuned",
    "report_to": ["tensorboard"],
    "overwrite_output_dir": True,
    "output_dir": "./tmp/vits_finetuned_bau",

    "dataset_name": "TON-PSEUDO-HF/bau-tts-monospeaker",
    "audio_column_name": "audio",
    "text_column_name": "text",
    "train_split_name": "train",
    "eval_split_name": "validation",

    "full_generation_sample_text": "Kɛ ɔ fɛ i aeroport Félix Houphouet-Boigny su lele mon fa ju",

    "max_duration_in_seconds": 30,
    "min_duration_in_seconds": 1.0,
    "max_tokens_length": 500,

    "model_name_or_path": "TON-PSEUDO-HF/mms-tts-bau-baseline",
    "tokenizer_name": "TON-PSEUDO-HF/mms-tts-bau-tokenizer",
    "override_vocabulary_embeddings": true,

    "preprocessing_num_workers": 4,

    "do_train": true,
    "num_train_epochs": 200,
    "gradient_accumulation_steps": 1,
    "gradient_checkpointing": false,
    "per_device_train_batch_size": 8,
    "learning_rate": 2e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_ratio": 0.01,
    "group_by_length": false,

    "do_eval": true,
    "eval_steps": 50,
    "per_device_eval_batch_size": 8,
    "max_eval_samples": 25,
    "do_step_schedule_per_epoch": true,

    "weight_disc": 3,
    "weight_fmaps": 1,
    "weight_gen": 1,
    "weight_kl": 1.5,
    "weight_duration": 1,
    "weight_mel": 35,

    "fp16": true,
    "seed": 456
}

import json
with open("finetune_baoule.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

# Étape 9 — Conversion du checkpoint donneur (akan → baseline baoulé)
Télécharge le discriminateur original (`facebook/mms-tts`, akan) et le générateur `facebook/mms-tts-aka`, corrige le `config.json` (ajoute `pad_token_id: 0`, absent par défaut et provoquant une `AttributeError`), puis lance le script officiel de conversion (fusion générateur + discriminateur) qui publie `mms-tts-bau-baseline` — le point de départ du fine-tuning.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import json, os, shutil

# 1. Récupère le discriminateur (déjà en cache depuis ta tentative précédente, donc quasi instantané)
checkpoint_path = hf_hub_download(
    repo_id="facebook/mms-tts",
    subfolder="full_models/aka",
    filename="D_100000.pth"
)
print("Discriminateur :", checkpoint_path)

# 2. Télécharge le générateur akan et copie-le dans un dossier modifiable
generator_cache_path = snapshot_download(repo_id="facebook/mms-tts-aka")
patched_path = "./mms-tts-aka-patched"
shutil.copytree(generator_cache_path, patched_path, dirs_exist_ok=True)

# 3. Patch : ajoute le pad_token_id manquant dans config.json
config_path = os.path.join(patched_path, "config.json")
with open(config_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)
cfg["pad_token_id"] = 0
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)

print("Config akan patchée dans :", patched_path)

In [ ]:
!python /content/finetune-hf-vits/convert_original_discriminator_checkpoint.py \
  --checkpoint_path "{checkpoint_path}" \
  --generator_checkpoint_path "{patched_path}" \
  --pytorch_dump_folder_path ./mms-tts-bau-baseline \
  --push_to_hub Tree-AI-lab/mms-tts-bau-baseline